# Fresh Kaggle T4 x2 Run: Fix 2 Only

Use a fresh Kaggle notebook/session with **Internet ON** and **Accelerator: T4 x2**. This notebook runs only Fix 2: the SQuAD n=500 x 5 seed headline-cell rigor upgrade.

Expected output files after packaging:

- `/kaggle/working/fix2_t4x2_outputs.zip`
- `/kaggle/working/AAA_FIX2_T4X2_OUTPUTS.zip`

The `AAA_...` copy is just there to make Kaggle's right-sidebar file browser easier to spot.

In [ ]:
import os
import pathlib
import subprocess
import time

WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'rag-hallucination-detection'
REMOTE = 'https://github.com/Saket-Maganti/rag-hallucination-detection.git'

def run(cmd, cwd=None, check=True):
    print('\n>>>', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check)

print('START', time.ctime(), flush=True)
run(['bash', '-lc', 'pkill -f kaggle_fix2_t4x2 || true; pkill -f kaggle_stream_fix2_t4x2 || true; pkill -x ollama || true'], check=False)
run(['nvidia-smi'], check=False)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--progress', '--branch', 'main', REMOTE, str(REPO)], cwd=WORK)
else:
    run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=False)
    run(['git', '-C', str(REPO), 'checkout', 'main'], check=False)
    run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=False)

os.chdir(REPO)
run(['git', 'log', '-1', '--oneline'])
run(['bash', '-lc', 'test -f scripts/kaggle_fix2_t4x2.sh && test -f scripts/kaggle_stream_fix2_t4x2.py && echo scripts OK'])
print('Repo ready at', REPO, flush=True)

## 1. Setup + Ollama Preflight

This installs dependencies, installs/verifies Ollama, starts two Ollama servers, pulls `mistral`, and checks both GPU ports. Wait for `preflight OK` before running the next cell.

In [ ]:
!python -u scripts/kaggle_stream_fix2_t4x2.py --stage setup --heartbeat 15

## 2. Run Fix 2 in Parallel

GPU0 runs seeds `41 42 43` on Ollama port `11434`. GPU1 runs seeds `44 45` on Ollama port `11435`. The wrapper prints heartbeat row counts and tails both shard logs.

In [ ]:
!python -u scripts/kaggle_stream_fix2_t4x2.py --stage parallel --heartbeat 30

## 3. Status

Use this after the parallel run, or after any interruption. It only prints row counts and output paths.

In [ ]:
!python -u scripts/kaggle_stream_fix2_t4x2.py --stage status --heartbeat 15

## 4. Package Outputs

Download `/kaggle/working/AAA_FIX2_T4X2_OUTPUTS.zip` or `/kaggle/working/fix2_t4x2_outputs.zip` from the Kaggle right sidebar after this completes.

In [ ]:
!python -u scripts/kaggle_stream_fix2_t4x2.py --stage package --heartbeat 15
!ls -lh /kaggle/working/fix2_t4x2_outputs.zip /kaggle/working/AAA_FIX2_T4X2_OUTPUTS.zip
!find /kaggle/working -maxdepth 2 \( -name '*FIX2*.zip' -o -name '*fix2*.zip' \) -print

## Debug If Anything Fails

Run this only if setup or the parallel run fails.

In [ ]:
!echo '--- git ---'
!git -C /kaggle/working/rag-hallucination-detection log -1 --oneline || true
!echo '--- zip files ---'
!find /kaggle/working -maxdepth 3 \( -name '*FIX2*.zip' -o -name '*fix2*.zip' \) -print -exec ls -lh {} \; || true
!echo '--- rows ---'
!python - <<'PY'
from pathlib import Path
import csv
base = Path('/kaggle/working/rag-hallucination-detection/data/revision/fix_02')
for p in sorted(base.glob('*.csv')):
    try:
        with p.open(newline='') as f:
            n = max(0, sum(1 for _ in csv.reader(f)) - 1)
    except Exception as exc:
        n = f'ERR:{exc}'
    print(p, n)
PY
!echo '--- processes ---'
!ps aux | grep -E 'ollama|fix2|kaggle_stream' | grep -v grep || true
!echo '--- gpu ---'
!nvidia-smi || true
!echo '--- wrapper ---'
!tail -n 120 /kaggle/working/fix2_t4x2_wrapper.log 2>/dev/null || true
!echo '--- gpu0 log ---'
!tail -n 160 /kaggle/working/rag-hallucination-detection/logs/revision/fix_02_gpu0_kaggle_t4x2.log 2>/dev/null || true
!echo '--- gpu1 log ---'
!tail -n 160 /kaggle/working/rag-hallucination-detection/logs/revision/fix_02_gpu1_kaggle_t4x2.log 2>/dev/null || true
!echo '--- ollama gpu0 ---'
!tail -n 120 /kaggle/working/fix2_t4x2_logs/ollama_gpu0.log 2>/dev/null || true
!echo '--- ollama gpu1 ---'
!tail -n 120 /kaggle/working/fix2_t4x2_logs/ollama_gpu1.log 2>/dev/null || true